In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch as t
from transformer_lens import HookedTransformer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from dadapy.data import Data
from collections import defaultdict


import transformer_lens.utils as utils
import einops

from joblib import Parallel, delayed
import pandas as pd

import plot_utils

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)


In [3]:
# Load GPT-2 Small
model = HookedTransformer.from_pretrained("gpt2-small")

# Load prompt from Pile-10K (Prompt 3218)
pile_dataset = load_dataset("NeelNanda/pile-10k")

Loaded pretrained model gpt2-small into HookedTransformer


In [4]:
filtered_indices = np.load('filtered_indices.npy')


In [5]:
filtered_indices

array([   0,   19,   22, ..., 9989, 9991, 9998], dtype=int64)

In [6]:
pile_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 10000
    })
})

In [7]:
filtered_dataset = pile_dataset['train'][filtered_indices]

In [8]:
setname_to_indexlist = defaultdict(list)

In [9]:
for i, set_name in enumerate(filtered_dataset['meta']):
    setname_to_indexlist[set_name['pile_set_name']].append(i)

In [10]:
arxiv_indices, wiki_indices, math_indices = setname_to_indexlist['ArXiv'], setname_to_indexlist['Wikipedia (en)'], setname_to_indexlist['DM Mathematics']

In [11]:
arxiv_prompts, wiki_prompts, math_prompts = [filtered_dataset['text'][idx] for idx in arxiv_indices], [filtered_dataset['text'][idx] for idx in wiki_indices], [filtered_dataset['text'][idx] for idx in math_indices]


In [12]:
def prompts_to_tokens(prompts):
    tokens = model.to_tokens(prompts, prepend_bos=True)
    tokens = tokens[..., :512]
    return tokens

In [13]:
arxiv_tokens, wiki_tokens, math_tokens = prompts_to_tokens(arxiv_prompts), prompts_to_tokens(wiki_prompts), prompts_to_tokens(math_prompts)

In [14]:
# Function to compute intrinsic dimension (ID) with dadapy
def compute_ids(full_reps):
    ids = []
    for rep in full_reps:
        _data = Data(coordinates=rep.cpu().detach().numpy(), maxk=100)
        ids.append(_data.return_id_scaling_gride(range_max=64))
    return np.array(ids)

In [15]:
def zero_attn_out_hook(attn_out, hook):
    # print(attn_out.shape)
    return t.zeros_like(attn_out)

zero_attn_hooks = [
            # Changed hook point to blocks.{layer}.attn.hook_result
            (utils.get_act_name("attn_out", layer), zero_attn_out_hook)
            for layer in range(model.cfg.n_layers)
        ]

In [16]:
def tokens_to_idim_prompt_and_free(tokens, N):
    logits, cache = model.run_with_cache(tokens[:N])
    accumulated_residual, labels = cache.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)
    with model.hooks(fwd_hooks=zero_attn_hooks):
        logits_free, cache_free = model.run_with_cache(tokens[:N])
        accumulated_residual_free, _ = cache_free.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)
        
        
    idim_accumulated = []
    idim_accumulated_free = []
    
    
    for idx in range(len(arxiv_tokens[:N])):
        ids = compute_ids(accumulated_residual[:, idx])
        ids_free = compute_ids(accumulated_residual_free[:, idx])
        idim_accumulated.append(ids[:, 0, 1])
        idim_accumulated_free.append(ids_free[:, 0, 1])
    # print(accumulated_residual.shape)
    return labels, idim_accumulated, idim_accumulated_free
        

In [25]:
def get_interaction_values(idim, idim_free):
    idim_interaction = np.array(idim_free) - np.array(idim)
    g_interaction = idim_interaction / np.array(idim)
    return idim_interaction, g_interaction

In [26]:
labels, idim, idim_free = tokens_to_idim_prompt_and_free(arxiv_tokens, 10)
idim_interaction, g_interaction = get_interaction_values(idim, idim_free)


In [27]:
labels, idim_wiki, idim_wiki_free = tokens_to_idim_prompt_and_free(wiki_tokens, 10)
idim_wiki_interaction, g_wiki_interaction = get_interaction_values(idim_wiki, idim_wiki_free)


In [28]:
labels, idim_math, idim_math_free = tokens_to_idim_prompt_and_free(math_tokens, 10)
idim_math_interaction, g_math_interaction = get_interaction_values(idim_math, idim_math_free)


In [20]:
from plot_utils import lines

In [21]:
import plotly.graph_objects as go

In [22]:
def summary_statistics(idim, n_bootstrap=1000):
    y_data = np.array(idim)
    
    num_lines, num_points = y_data.shape
    # --- 2. Calculate Average and Bootstrap Standard Error ---

    # Calculate the average y-value for each x-point across all 10 lines.
    y_average = np.mean(y_data, axis=0)

    # Calculate error bars using bootstrapping
    bootstrap_std_err = np.zeros(num_points)

    for j in range(num_points): # For each x-point (column)
        point_data = y_data[:, j] # Get the y-values from all lines for this x-point
        bootstrap_means = np.zeros(n_bootstrap)

        for i in range(n_bootstrap):
            # Resample the data *with replacement*
            resampled_data = np.random.choice(point_data, size=num_lines, replace=True)
            # Calculate the mean of the resampled data
            bootstrap_means[i] = np.mean(resampled_data)

        # Calculate the standard deviation of the bootstrap means
        # This serves as the bootstrapped estimate of the standard error.
        # Alternatively, one could calculate a confidence interval (e.g., 2.5th and 97.5th percentiles)
        # from bootstrap_means for potentially asymmetric error bars.
        bootstrap_std_err[j] = np.std(bootstrap_means, ddof=1)
    return y_average, bootstrap_std_err
        

In [32]:
arxiv_prompt_average, arxiv_prompt_err = summary_statistics(idim)
arxiv_free_average, arxiv_free_err = summary_statistics(idim_free)

arxiv_interaction_average, arxiv_interaction_err = summary_statistics(idim_interaction)
arxiv_g_average, arxiv_g_err = summary_statistics(g_interaction)

In [33]:
wiki_prompt_average, wiki_prompt_err = summary_statistics(idim_wiki)
wiki_free_average, wiki_free_err = summary_statistics(idim_wiki_free)

wiki_interaction_average, wiki_interaction_err = summary_statistics(idim_wiki_interaction)
wiki_g_average, wiki_g_err = summary_statistics(g_wiki_interaction)

In [34]:
math_prompt_average, math_prompt_err = summary_statistics(idim_math)
math_free_average, math_free_err = summary_statistics(idim_math_free)

math_interaction_average, math_interaction_err = summary_statistics(idim_math_interaction)
math_g_average, math_g_err = summary_statistics(g_math_interaction)

In [36]:
import plotly
from plotly.colors import qualitative
colors = qualitative.Plotly # Get the default sequence
D3colors = qualitative.D3

In [37]:
def get_alpha_color_string(hex_color, alpha = 0.5):
    rgb_tuple = plotly.colors.hex_to_rgb(hex_color)
    return f'rgba({rgb_tuple[0]}, {rgb_tuple[1]}, {rgb_tuple[2]}, {alpha})'

In [29]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_prompt_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv Prompt',   # Name for the legend
    line=dict(color=colors[0], width=2), # Style the average line
    marker=dict(size=5, color=colors[0]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_prompt_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the e rror bar caps
        color=get_alpha_color_string(colors[0]) # Color of error bars (royalblue with some transparency)
    )
))

fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_free_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv Free',   # Name for the legend
    line=dict(color=D3colors[0], width=2), # Style the average line
    marker=dict(size=5, color=D3colors[0]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_free_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(D3colors[0]) # Color of error bars (royalblue with some transparency)
    )
))

fig.add_trace(go.Scatter(
    x=labels,
    y=wiki_prompt_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Wiki Prompt',   # Name for the legend
    line=dict(color=colors[1], width=2), # Style the average line
    marker=dict(size=5, color=colors[1]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=wiki_prompt_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[1]) # Color of error bars (royalblue with some transparency)
    )
))

fig.add_trace(go.Scatter(
    x=labels,
    y=wiki_free_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Wiki Free',   # Name for the legend
    line=dict(color=D3colors[1], width=2), # Style the average line
    marker=dict(size=5, color=D3colors[1]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=wiki_free_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(D3colors[1]) # Color of error bars (royalblue with some transparency)
    )
))

fig.add_trace(go.Scatter(
    x=labels,
    y=math_prompt_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Math Prompt',   # Name for the legend
    line=dict(color=colors[2], width=2), # Style the average line
    marker=dict(size=5, color=colors[2]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=math_prompt_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[2]) # Color of error bars (royalblue with some transparency)
    )
))

fig.add_trace(go.Scatter(
    x=labels,
    y=math_free_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Math Free',   # Name for the legend
    line=dict(color=D3colors[2], width=2), # Style the average line
    marker=dict(size=5, color=D3colors[2]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=math_free_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(D3colors[2]) # Color of error bars (royalblue with some transparency)
    )
))

# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=22)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [38]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_interaction_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv interaction',   # Name for the legend
    line=dict(color=colors[0], width=2), # Style the average line
    marker=dict(size=5, color=colors[0]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_interaction_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the e rror bar caps
        color=get_alpha_color_string(colors[0]) # Color of error bars (royalblue with some transparency)
    )
))


fig.add_trace(go.Scatter(
    x=labels,
    y=wiki_interaction_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Wiki interaction',   # Name for the legend
    line=dict(color=colors[1], width=2), # Style the average line
    marker=dict(size=5, color=colors[1]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=wiki_interaction_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[1]) # Color of error bars (royalblue with some transparency)
    )
))



fig.add_trace(go.Scatter(
    x=labels,
    y=math_interaction_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Math interaction',   # Name for the legend
    line=dict(color=colors[2], width=2), # Style the average line
    marker=dict(size=5, color=colors[2]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=math_interaction_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[2]) # Color of error bars (royalblue with some transparency)
    )
))


# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=22)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [39]:
# Initialize a Plotly Figure object
fig = go.Figure()



# Example: Get the first color


# Add the main trace: the average line with error bars
fig.add_trace(go.Scatter(
    x=labels,
    y=arxiv_g_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Arxiv g',   # Name for the legend
    line=dict(color=colors[0], width=2), # Style the average line
    marker=dict(size=5, color=colors[0]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=arxiv_g_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the e rror bar caps
        color=get_alpha_color_string(colors[0]) # Color of error bars (royalblue with some transparency)
    )
))


fig.add_trace(go.Scatter(
    x=labels,
    y=wiki_g_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Wiki g',   # Name for the legend
    line=dict(color=colors[1], width=2), # Style the average line
    marker=dict(size=5, color=colors[1]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=wiki_g_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[1]) # Color of error bars (royalblue with some transparency)
    )
))



fig.add_trace(go.Scatter(
    x=labels,
    y=math_g_average,
    mode='lines+markers',  # Display both the line connecting points and markers at each point
    name='Math g',   # Name for the legend
    line=dict(color=colors[2], width=2), # Style the average line
    marker=dict(size=5, color=colors[2]), # Style the markers
    error_y=dict(
        type='data',       # Error bar type: 'data' means the 'array' values are the error magnitudes
        array=math_g_err,   # Provide the calculated standard error values for the error bars
        visible=True,      # Make the error bars visible
        thickness=1,       # Thickness of the error bar lines
        width=3,           # Width of the error bar caps
        color=get_alpha_color_string(colors[2]) # Color of error bars (royalblue with some transparency)
    )
))


# --- Optional: Add the original individual lines for context (commented out by default) ---
# Uncomment the following loop if you want to see the underlying data lines.
# for i in range(num_lines):
#     fig.add_trace(go.Scatter(
#         x=x_values,
#         y=y_data[i, :],
#         mode='lines',
#         name=f'Line {i+1}',
#         line=dict(width=0.5, color='rgba(128, 128, 128, 0.4)'), # Thin, semi-transparent grey lines
#         showlegend=False # Hide these individual lines from the main legend
#     ))

# --- 4. Customize the Plot Layout ---
fig.update_layout(font=dict(size=22)) 
fig.update_layout(
    title='Average of 10 Prompts', # Plot title
    xaxis_title='Layers',                 # X-axis label
    yaxis_title='IDim',                               # Y-axis label
    legend_title='Prompt and IDim Type',                               # Title for the legend box
    hovermode='x unified',                               # Show hover info for all traces at a given x-value
    template='plotly_white',
    showlegend=True,
)

#fig.write_image('IDIM_prompt_categories.png')
# --- 5. Show the Plot ---

# Display the figure. This will typically open it in a browser window
# or display it in the output cell of a Jupyter notebook.
fig.show()


In [24]:
lines(idim_free,x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels, names = list(range(10)))


In [23]:
lines(idim,x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels, names = list(range(10)))



In [42]:
idim_accumulated

[array([3.47, 5.2 , 4.37, 5.02, 4.69, 6.14, 5.93, 6.88, 6.87, 7.47, 7.44,
        8.11, 7.71, 8.09, 7.78, 7.89, 7.55, 7.73, 7.46, 7.52, 7.42, 7.39,
        7.42, 7.44, 7.49])]

In [16]:
ids.shape

(25, 3, 6)

In [55]:
per_layer_residual, labels = cache.decompose_resid(
    layer=-1, return_labels=True
)
print(labels)
print(per_layer_residual.shape)
# ids = compute_ids(per_layer_residual.squeeze())
# line(ids[1:, 0, 1], hover_name=labels[1:], title="ID from Per-Layer Residuals", x=np.arange(model.cfg.n_layers * 2 + 1) / 2)
# print(ids.shape)
# lines(ids[:, 0, :3].T, hover_name=labels, x=np.arange(model.cfg.n_layers * 2 + 2) / 2, names = ["2", "4", "8"])
idim_per_layer = []
for idx in range(len(tokens)):
  ids = compute_ids(per_layer_residual[:, idx])
  # print(accumulated_residual.shape)
  idim_per_layer.append(ids[1:, 0, 1])
  # line(ids[:, 0, 1],x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels)

lines(idim_per_layer,x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels[1:], names = ['Prompt 3218'])

['embed', 'pos_embed', '0_attn_out', '0_mlp_out', '1_attn_out', '1_mlp_out', '2_attn_out', '2_mlp_out', '3_attn_out', '3_mlp_out', '4_attn_out', '4_mlp_out', '5_attn_out', '5_mlp_out', '6_attn_out', '6_mlp_out', '7_attn_out', '7_mlp_out', '8_attn_out', '8_mlp_out', '9_attn_out', '9_mlp_out', '10_attn_out', '10_mlp_out', '11_attn_out', '11_mlp_out']
torch.Size([26, 1, 512, 768])


c:\Users\saepa\anaconda3\envs\arena-env\Lib\site-packages\dadapy\id_estimation.py:413: UserWarning:

there may be data with zero distance from each other;
                this may compromise the correct behavior of some routines



In [18]:
per_head_residual, labels = cache.stack_head_results(
    layer=-1, return_labels=True, incl_remainder=True
)

Tried to stack head results when they weren't cached. Computing head results now


In [19]:
per_head_accumulate= t.cumsum(per_head_residual, dim=0)

In [20]:

# ids = compute_ids(per_layer_residual.squeeze())
# line(ids[1:, 0, 1], hover_name=labels[1:], title="ID from Per-Layer Residuals", x=np.arange(model.cfg.n_layers * 2 + 1) / 2)
# print(ids.shape)
# lines(ids[:, 0, :3].T, hover_name=labels, x=np.arange(model.cfg.n_layers * 2 + 2) / 2, names = ["2", "4", "8"])
tensors = []
for idx in range(len(tokens)):
  ids = compute_ids(per_head_accumulate[:, idx])
  tensors.append(ids[:, 0, 1])
  # print(accumulated_residual.shape)
  #tensors.append(ids[1:, 0, 1])
  # line(ids[:, 0, 1],x=np.arange(model.cfg.n_layers * 2 + 1) / 2, hover_name=labels)

lines(tensors,x=labels, hover_name=labels, names = ['Prompt 3218'])

In [21]:
per_head_residual.shape

torch.Size([145, 1, 512, 768])

In [22]:

for idx in range(len(tokens)):
  ids = compute_ids(per_head_residual[1:, idx])
  # print(accumulated_residual.shape)
  # tensors.append(ids[:, 0, 1])
  ids = ids[:, 0, 1]
  ids = einops.rearrange(
      ids,
      "(layer head_index) -> layer head_index",
      layer=model.cfg.n_layers,
      head_index=model.cfg.n_heads,
  )
  imshow(
      ids,
      labels={"x": "Head", "y": "Layer"},
      title="IDIM",
  )



In [23]:
accumulated_residual.shape # layers, batch, ntokens, embed

torch.Size([25, 1, 512, 768])

In [24]:
model.tokens_to_residual_directions

<bound method HookedTransformer.tokens_to_residual_directions of HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_

In [25]:
layers = model.cfg.n_layers

# 1. ID from accumulated_resid (pre and mid at each layer)
accumulated_residual, labels = cache.accumulated_resid(layer = -1, incl_mid=True,\
                                                return_labels = True)

In [26]:
def get_scaled_residual_stack_logits(
    residual_stack,
    cache,
) -> float:
    scaled_residual_stack = cache.apply_ln_to_stack(
        residual_stack, layer=-1, pos_slice=-1
    )
    return einops.einsum(scaled_residual_stack, model.W_U, "layers batch ntokens emb, emb vocab -> layers batch ntokens vocab")

In [27]:
logits = get_scaled_residual_stack_logits(accumulated_residual, cache)

In [28]:
logits.shape

torch.Size([25, 1, 512, 50257])

In [29]:
loss = []
for logit in logits:
    loss.append(utils.lm_cross_entropy_loss(logit, tokens).item())
    #print(utils.lm_cross_entropy_loss(logit, tokens))

In [30]:
len(tensors)

1

In [31]:
line(
    loss,
    x=np.arange(model.cfg.n_layers * 2 + 1) / 2,
    hover_name=labels,
    title="Loss From Accumulate Residual Stream, Prompt 3218",
)

In [53]:
per_layer_residual, labels = cache.decompose_resid(
    layer=-1, return_labels=True
)

In [54]:
labels

['embed',
 'pos_embed',
 '0_attn_out',
 '0_mlp_out',
 '1_attn_out',
 '1_mlp_out',
 '2_attn_out',
 '2_mlp_out',
 '3_attn_out',
 '3_mlp_out',
 '4_attn_out',
 '4_mlp_out',
 '5_attn_out',
 '5_mlp_out',
 '6_attn_out',
 '6_mlp_out',
 '7_attn_out',
 '7_mlp_out',
 '8_attn_out',
 '8_mlp_out',
 '9_attn_out',
 '9_mlp_out',
 '10_attn_out',
 '10_mlp_out',
 '11_attn_out',
 '11_mlp_out']

In [33]:
accumulated_residual.shape

torch.Size([25, 1, 512, 768])

In [34]:
per_layer_residual.shape

torch.Size([26, 1, 512, 768])

In [35]:
logits_per_layer = get_scaled_residual_stack_logits(per_layer_residual, cache)
loss_per_layer = []
for logit in logits_per_layer:
    loss_per_layer.append(utils.lm_cross_entropy_loss(logit, tokens).item())
    #print(utils.lm_cross_entropy_loss(logit, tokens))

In [36]:
len(labels)

26

In [37]:
len(np.arange(model.cfg.n_layers * 2 + 1) / 2 )

25

In [38]:
line(
    loss_per_layer,
    x=labels ,
    hover_name=labels,
    title="Loss From Each Layer, Prompt 3218",
)

In [59]:
fig = plot_utils.create_correlation_plot(
    idim_accumulated[0], loss, 
    title="Accumulated Residual",
    x_label="IDim",
    y_label="Loss"
)
fig.update_layout(font_size=20)
fig.show()

In [51]:
len(idim_per_layer[0])

25

In [52]:
len(loss_per_layer)

26

In [56]:
fig = plot_utils.create_correlation_plot(
    idim_per_layer[0], loss_per_layer[1:], 
    title="Per Layer",
    x_label="IDim",
    y_label="Loss"
)
fig.update_layout(font_size=20)

fig.show()